# Tutorial 3: QAOA for Portfolio Optimization

This notebook solves the cardinality-constrained portfolio QUBO using the
Quantum Approximate Optimization Algorithm (QAOA). We cover the cost unitary,
mixer unitary, parameter optimization, and result interpretation.

**Reference**: Farhi, Goldstone, Gutmann (2014). Brandhofer et al. (2023).

In [ ]:
import numpy as np
np.random.seed(42)

## 1. Build the Portfolio QUBO

In [ ]:
from qufin.portfolio.qubo import PortfolioQUBO

mu = np.array([0.12, 0.10, 0.07, 0.03, 0.15])
cov = np.array([
    [0.040, 0.006, 0.002, 0.000, 0.010],
    [0.006, 0.030, 0.004, 0.001, 0.008],
    [0.002, 0.004, 0.020, 0.002, 0.003],
    [0.000, 0.001, 0.002, 0.010, 0.001],
    [0.010, 0.008, 0.003, 0.001, 0.050],
])

qubo = PortfolioQUBO(mu=mu, cov=cov, gamma=0.5, cardinality=2)
print(f"QUBO with {qubo.n_assets} assets, K={qubo.cardinality}, {qubo.n_qubits} qubits")

## 2. QAOA with X Mixer (Standard)

The standard QAOA uses $R_X$ mixers. Cardinality is enforced via penalty terms.

In [ ]:
from qufin.portfolio.optimizers.qaoa import QAOAPortfolio, QAOAConfig
from qufin.backends.qiskit_backend import QiskitAerBackend

backend = QiskitAerBackend(method="automatic", seed=42)  # shots set per QAOAConfig

config_x = QAOAConfig(
    p=2,              # QAOA depth (number of layers)
    mixer="x",        # Standard X mixer
    optimizer="COBYLA",
    maxiter=100,
    shots=4096,
    seed=42,
)

solver_x = QAOAPortfolio(qubo, config_x, backend)
result_x = solver_x.run()

print(f"X mixer result:")
print(f"  Best bitstring: {result_x.best_bitstring}")
print(f"  Objective:      {result_x.best_objective:.6f}")
print(f"  Feasible:       {result_x.feasible}")

## 3. QAOA with XY-Ring Mixer

The XY-ring mixer preserves Hamming weight, so starting from a Dicke state
with weight K, the search is restricted to feasible solutions (exactly K assets).
This eliminates the need for penalty terms.

In [ ]:
config_xy = QAOAConfig(
    p=2,
    mixer="xy_ring",  # XY-ring preserves Hamming weight
    cardinality=2,
    optimizer="COBYLA",
    maxiter=100,
    shots=4096,
    seed=42,
)

solver_xy = QAOAPortfolio(qubo, config_xy, backend)
result_xy = solver_xy.run()

print(f"XY-ring mixer result:")
print(f"  Best bitstring: {result_xy.best_bitstring}")
print(f"  Objective:      {result_xy.best_objective:.6f}")
print(f"  Feasible:       {result_xy.feasible}")

## 4. Effect of QAOA Depth (p)

Deeper QAOA circuits can find better solutions but are harder to optimize.

In [ ]:
for p in [1, 2, 3, 4]:
    config = QAOAConfig(
        p=p, mixer="xy_ring", cardinality=2,
        optimizer="COBYLA", maxiter=150, shots=4096, seed=42,
    )
    solver = QAOAPortfolio(qubo, config, backend)
    result = solver.run()
    print(f"  p={p}: obj={result.best_objective:.6f}  bits={result.best_bitstring}  feasible={result.feasible}")

## 5. CVaR Objective

Using CVaR (Conditional Value at Risk) as the aggregation function focuses
optimization on the best samples from the measurement distribution.

In [ ]:
config_cvar = QAOAConfig(
    p=2, mixer="xy_ring", cardinality=2,
    optimizer="COBYLA", maxiter=100, shots=4096,
    cvar_alpha=0.25,  # Use only the best 25% of samples
    seed=42,
)

solver_cvar = QAOAPortfolio(qubo, config_cvar, backend)
result_cvar = solver_cvar.run()

print(f"CVaR QAOA result:")
print(f"  Best bitstring: {result_cvar.best_bitstring}")
print(f"  Objective:      {result_cvar.best_objective:.6f}")

## 6. Compare with Classical Optimal

In [ ]:
from itertools import combinations

Q = qubo.build_matrix()
n = len(mu)

# Brute-force optimal
best_cost = float("inf")
best_bits = None
for combo in combinations(range(n), 2):
    x = np.zeros(n)
    x[list(combo)] = 1.0
    cost = x @ Q @ x
    if cost < best_cost:
        best_cost = cost
        best_bits = "".join(str(int(b)) for b in x)

print(f"Classical optimal:   {best_bits}  cost={best_cost:.6f}")
print(f"QAOA (XY-ring, p=2): {result_xy.best_bitstring}  cost={result_xy.best_objective:.6f}")
approx_ratio = best_cost / result_xy.best_objective if result_xy.best_objective != 0 else 0
print(f"Approximation ratio: {approx_ratio:.4f}")

## Summary

In this tutorial we covered:
- QAOA with standard X mixer and penalty-based cardinality
- XY-ring mixer for hard cardinality enforcement
- Effect of QAOA depth (p) on solution quality
- CVaR objective for focused optimization
- Comparison with brute-force classical optimal

**Next**: Tutorial 04 uses VQE with hardware-efficient ansatz for the same problem.